# 🎮 Task 1: Web Scraping — Data Collection Steam

This notebook uses **Steam Web API** (no API key needed) to collect:
- Game info: Name, genres, developer, price
- User reviews: Comment content, Recommended/Not Recommended label

**Topic:** *Resident Evil* Series on Steam (7 games)

**Output:** `../data/raw_steam_data.csv` and `../data/raw_games_info.csv`

---

### API usage
| API | Endpoint |
|-----|----------|
| Game Details | `https://store.steampowered.com/api/appdetails?appids={appid}` |
| Reviews | `https://store.steampowered.com/appreviews/{appid}?json=1` |

> ⚠️ **Estimated time:** ~25–40 minutes (7 game × 500 reviews × ~1.2s delay)

In [1]:
# ============================================================
# Cell 1 — Import libraries
# ============================================================
import requests
import pandas as pd
import time
import os
from datetime import datetime

print('Libraries loaded successfully!')
print(f'  pandas   : {pd.__version__}')
print(f'  requests : {requests.__version__}')

Libraries loaded successfully!
  pandas   : 3.0.3
  requests : 2.34.2


In [2]:
# ============================================================
# Cell 2 — Configuration: Game list target
# ============================================================

GAMES = [
    {'appid': 418370,  'name': 'Resident Evil 7 Biohazard'},
    {'appid': 883710,  'name': 'Resident Evil 2 (2019)'},
    {'appid': 952060,  'name': 'Resident Evil 3 (2020)'},
    {'appid': 1196590, 'name': 'Resident Evil Village'},
    {'appid': 1880560, 'name': 'Resident Evil 4 (2023)'},
    {'appid': 21690,   'name': 'Resident Evil 5'},
    {'appid': 221040,  'name': 'Resident Evil 6'},
]

REVIEWS_PER_GAME = 500   # Increase/decrease as needed
DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

print('=== Configuration ===')
print(f'  Target games count     : {len(GAMES)}')
print(f'  Reviews per game   : {REVIEWS_PER_GAME}')
print(f'  Estimated total reviews: ~{len(GAMES) * REVIEWS_PER_GAME}')
print(f'  Output directory     : {os.path.abspath(DATA_DIR)}')
print()
print('Game list:')
for g in GAMES:
    print(f'  [{g["appid"]}] {g["name"]}')

=== Configuration ===
  Target games count     : 7
  Reviews per game   : 500
  Estimated total reviews: ~3500
  Output directory     : D:\qkhenh\Project\CodeAlpha_DataAnalytics\data

Game list:
  [418370] Resident Evil 7 Biohazard
  [883710] Resident Evil 2 (2019)
  [952060] Resident Evil 3 (2020)
  [1196590] Resident Evil Village
  [1880560] Resident Evil 4 (2023)
  [21690] Resident Evil 5
  [221040] Resident Evil 6


In [3]:
# ============================================================
# Cell 3 — Function to get detailed game info
# ============================================================

def get_game_details(appid):
    """Get game metadata from Steam Store API."""
    url = 'https://store.steampowered.com/api/appdetails'
    params = {'appids': appid, 'cc': 'us', 'l': 'english'}
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; SteamAnalytics/1.0)'}

    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
        resp.raise_for_status()
        payload = resp.json()

        game_node = payload.get(str(appid), {})
        if not game_node.get('success'):
            print(f'  [WARN] appid={appid}: API returned success=false')
            return None

        d = game_node['data']
        price_info = d.get('price_overview', {})

        return {
            'appid'       : appid,
            'name'        : d.get('name', ''),
            'developer'   : ', '.join(d.get('developers', [])),
            'publisher'   : ', '.join(d.get('publishers', [])),
            'genres'      : ', '.join([g['description'] for g in d.get('genres', [])]),
            'categories'  : ', '.join([c['description'] for c in d.get('categories', [])[:6]]),
            'price_usd'   : price_info.get('final', 0) / 100 if price_info else 0.0,
            'is_free'     : d.get('is_free', False),
            'release_date': d.get('release_date', {}).get('date', ''),
            'short_desc'  : d.get('short_description', '')[:200],
        }
    except requests.exceptions.RequestException as e:
        print(f'  [ERROR] Network error for appid={appid}: {e}')
        return None
    except Exception as e:
        print(f'  [ERROR] Unexpected error for appid={appid}: {e}')
        return None


# --- Quick test ---
print('Testing get_game_details() with Resident Evil 7...')
test = get_game_details(418370)
if test:
    print(f'  OK! Name     : {test["name"]}')
    print(f'       Genres  : {test["genres"]}')
    print(f'       Price   : ${test["price_usd"]:.2f}')
    print(f'       Release : {test["release_date"]}')
else:
    print('  FAILED — check internet connection')

Testing get_game_details() with Resident Evil 7...
  OK! Name     : Resident Evil 7 Biohazard
       Genres  : Action, Adventure
       Price   : $19.99
       Release : Jan 23, 2017


In [4]:
# ============================================================
# Cell 4 — Function to get reviews (with cursor pagination)
# ============================================================

def get_steam_reviews(appid, max_reviews=500, language='english'):
    """
    Get reviews from Steam Review API with cursor-based pagination.
    Returns a list of dicts, each dict is a review.
    """
    reviews = []
    cursor = '*'
    page = 0
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; SteamAnalytics/1.0)'}

    while len(reviews) < max_reviews:
        params = {
            'json'         : 1,
            'num_per_page' : 100,
            'cursor'       : cursor,
            'language'     : language,
            'filter'       : 'all',
            'review_type'  : 'all',
            'purchase_type': 'all',
        }

        try:
            url = f'https://store.steampowered.com/appreviews/{appid}'
            resp = requests.get(url, params=params, headers=headers, timeout=20)
            resp.raise_for_status()
            data = resp.json()

            if data.get('success') != 1:
                print(f'  [WARN] API success!=1 on page {page}')
                break

            batch = data.get('reviews', [])
            if not batch:
                break  # No more reviews

            for r in batch:
                author = r.get('author', {})
                reviews.append({
                    'appid'                  : appid,
                    'review_id'              : r.get('recommendationid', ''),
                    'author_steamid'         : author.get('steamid', ''),
                    'num_games_owned'        : author.get('num_games_owned', 0),
                    'num_reviews'            : author.get('num_reviews', 0),
                    'playtime_forever_mins'  : author.get('playtime_forever', 0),
                    'playtime_at_review_mins': author.get('playtime_at_review', 0),
                    'review_text'            : r.get('review', ''),
                    'voted_up'               : r.get('voted_up', False),
                    'votes_up'               : r.get('votes_up', 0),
                    'votes_funny'            : r.get('votes_funny', 0),
                    'weighted_vote_score'    : float(r.get('weighted_vote_score', 0)),
                    'timestamp_created'      : r.get('timestamp_created', 0),
                    'timestamp_updated'      : r.get('timestamp_updated', 0),
                    'steam_purchase'         : r.get('steam_purchase', False),
                    'early_access_review'    : r.get('written_during_early_access', False),
                    'language'               : r.get('language', ''),
                })

            # Cursor to get next page
            new_cursor = data.get('cursor', '')
            if not new_cursor or new_cursor == cursor:
                break
            cursor = new_cursor
            page += 1

            print(f'    Page {page:>3}: {len(reviews):>5} reviews...', end='\r')
            time.sleep(1.2)  # Rate limit — do not spam Steam

        except requests.exceptions.RequestException as e:
            print(f'  [WARN] Request error page {page}: {e}')
            time.sleep(5)
            break
        except Exception as e:
            print(f'  [ERROR] Unexpected: {e}')
            break

    return reviews[:max_reviews]


print('get_steam_reviews() is ready.')

get_steam_reviews() is ready.


In [5]:
# ============================================================
# Cell 5 — Main data collection loop
# ============================================================

all_reviews = []
all_game_details = []
scraping_log = []

start_time = datetime.now()
print(f'Started at: {start_time.strftime("%H:%M:%S")}')
print('=' * 60)

for i, game in enumerate(GAMES):
    appid = game['appid']
    name  = game['name']
    g_start = datetime.now()

    print(f'\n[{i+1}/{len(GAMES)}] {name}  (AppID: {appid})')
    print('-' * 60)

    # ---- 1. Game details ----
    print('  [1/2] Get game info...')
    details = get_game_details(appid)
    if details:
        all_game_details.append(details)
        print(f'        Genres   : {details["genres"]}')
        print(f'        Developer: {details["developer"]}')
        print(f'        Price    : ${details["price_usd"]:.2f}')
    else:
        print('        [SKIP] Could not get game details')
    time.sleep(1.5)

    # ---- 2. Reviews ----
    print(f'  [2/2] Get reviews (max {REVIEWS_PER_GAME})...')
    reviews = get_steam_reviews(appid, REVIEWS_PER_GAME)

    for r in reviews:
        r['game_name'] = name
    all_reviews.extend(reviews)

    elapsed = (datetime.now() - g_start).seconds
    log_entry = {
        'appid'             : appid,
        'name'              : name,
        'reviews_collected' : len(reviews),
        'elapsed_sec'       : elapsed,
        'status'            : 'OK' if reviews else 'NO_DATA',
    }
    scraping_log.append(log_entry)
    print(f'\n  Done: {len(reviews)} reviews | {elapsed}s')
    time.sleep(2.0)

total_elapsed = (datetime.now() - start_time).seconds // 60
print(f'\n{"="*60}')
print(f'Collection complete!')
print(f'  Total reviews  : {len(all_reviews)}')
print(f'  Total game info: {len(all_game_details)}')
print(f'  Time     : ~{total_elapsed} minutes')

Started at: 17:44:43

[1/7] Resident Evil 7 Biohazard  (AppID: 418370)
------------------------------------------------------------
  [1/2] Get game info...
        Genres   : Action, Adventure
        Developer: CAPCOM Co., Ltd.
        Price    : $19.99
  [2/2] Get reviews (max 500)...
    Page   5:   500 reviews...
  Done: 500 reviews | 10s

[2/7] Resident Evil 2 (2019)  (AppID: 883710)
------------------------------------------------------------
  [1/2] Get game info...
        Genres   : Action
        Developer: CAPCOM Co., Ltd.
        Price    : $39.99
  [2/2] Get reviews (max 500)...
    Page   5:   500 reviews...
  Done: 500 reviews | 10s

[3/7] Resident Evil 3 (2020)  (AppID: 952060)
------------------------------------------------------------
  [1/2] Get game info...
        Genres   : Action
        Developer: CAPCOM Co., Ltd.
        Price    : $39.99
  [2/2] Get reviews (max 500)...
    Page   5:   500 reviews...
  Done: 500 reviews | 13s

[4/7] Resident Evil Village  (A

In [6]:
# ============================================================
# Cell 6 — Create DataFrame and basic processing
# ============================================================

reviews_df = pd.DataFrame(all_reviews)
games_df   = pd.DataFrame(all_game_details)

# Convert Unix timestamp to datetime
reviews_df['date_created'] = pd.to_datetime(
    reviews_df['timestamp_created'], unit='s', errors='coerce'
)
reviews_df['date_updated'] = pd.to_datetime(
    reviews_df['timestamp_updated'], unit='s', errors='coerce'
)

# Add readable label
reviews_df['recommendation'] = reviews_df['voted_up'].map(
    {True: 'Recommended', False: 'Not Recommended'}
)

# Calculate review length
reviews_df['review_length'] = reviews_df['review_text'].str.len().fillna(0).astype(int)

# Convert playtime to hours
reviews_df['playtime_hours']          = (reviews_df['playtime_forever_mins'] / 60).round(1)
reviews_df['playtime_at_review_hours'] = (reviews_df['playtime_at_review_mins'] / 60).round(1)

print(f'Reviews DataFrame: {reviews_df.shape}')
print(f'Games DataFrame  : {games_df.shape}')
print()
print('Recommendation distribution:')
print(reviews_df['recommendation'].value_counts())
print()
print('Reviews by game:')
print(reviews_df['game_name'].value_counts())

Reviews DataFrame: (3000, 24)
Games DataFrame  : (6, 10)

Recommendation distribution:
recommendation
Recommended        2579
Not Recommended     421
Name: count, dtype: int64

Reviews by game:
game_name
Resident Evil 7 Biohazard    500
Resident Evil 2 (2019)       500
Resident Evil 3 (2020)       500
Resident Evil Village        500
Resident Evil 5              500
Resident Evil 6              500
Name: count, dtype: int64


In [7]:
# ============================================================
# Cell 7 — View sample data
# ============================================================

print('=== Sample Reviews DataFrame ===')
display_cols = ['game_name', 'recommendation', 'review_text', 'playtime_hours',
                'votes_up', 'date_created', 'review_length']
display(reviews_df[display_cols].head(5))

print()
print('=== Sample Games DataFrame ===')
display(games_df)

=== Sample Reviews DataFrame ===


,game_name,recommendation,review_text,playtime_hours,votes_up,date_created,review_length
0,Resident Evil 7 Biohazard,Recommended,Resident Evil 7 is so good it's actually stupi...,12.0,49,2026-05-30 10:11:38,678
1,Resident Evil 7 Biohazard,Recommended,"A little late to the party, but Resident Evil ...",15.8,17,2026-06-24 14:48:46,283
2,Resident Evil 7 Biohazard,Recommended,This is one of the best horror games I've ever...,40.8,12,2026-06-14 11:17:53,1263
3,Resident Evil 7 Biohazard,Recommended,Resident Evil 7: Biohazard is a strong return ...,31.4,14,2026-06-09 22:18:31,968
4,Resident Evil 7 Biohazard,Recommended,"Definitely good game, I do recommend it. story...",16.1,8,2026-06-06 11:28:48,368



=== Sample Games DataFrame ===


,appid,name,developer,publisher,genres,categories,price_usd,is_free,release_date,short_desc
0,418370,Resident Evil 7 Biohazard,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.","Action, Adventure","Single-player, Steam Achievements, Full contro...",19.99,False,"Jan 23, 2017",Fear and isolation seep through the walls of a...
1,883710,Resident Evil 2,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Action,"Single-player, Steam Achievements, Full contro...",39.99,False,"Jan 24, 2019",A deadly virus engulfs the residents of Raccoo...
2,952060,Resident Evil 3,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Action,"Single-player, Multi-player, PvP, Online PvP, ...",39.99,False,"Apr 2, 2020",Jill Valentine is one of the last remaining pe...
3,1196590,Resident Evil Village,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Action,"Single-player, Steam Achievements, Full contro...",39.99,False,"May 6, 2021",Experience survival horror like never before i...
4,21690,Resident Evil 5,Capcom,Capcom,"Action, Adventure","Single-player, Multi-player, Co-op, Steam Achi...",19.99,False,"Sep 15, 2009",This game is a port of the Games for Windows -...
5,221040,Resident Evil 6,Capcom,Capcom,"Action, Adventure","Single-player, Multi-player, Co-op, Shared/Spl...",19.99,False,"Mar 21, 2013","Blending action and survival horror, Resident ..."


In [8]:
# ============================================================
# Cell 8 — Scraping summary log
# ============================================================

log_df = pd.DataFrame(scraping_log)
print('=== Scraping Log ===')
display(log_df)
print()
print(f'Total reviews collected: {log_df["reviews_collected"].sum()}')
print(f'Game with fewest reviews : {log_df.loc[log_df["reviews_collected"].idxmin(), "name"]} ({log_df["reviews_collected"].min()})')

=== Scraping Log ===


,appid,name,reviews_collected,elapsed_sec,status
0,418370,Resident Evil 7 Biohazard,500,10,OK
1,883710,Resident Evil 2 (2019),500,10,OK
2,952060,Resident Evil 3 (2020),500,13,OK
3,1196590,Resident Evil Village,500,11,OK
4,1880560,Resident Evil 4 (2023),0,3,NO_DATA
5,21690,Resident Evil 5,500,11,OK
6,221040,Resident Evil 6,500,11,OK



Total reviews collected: 3000
Game with fewest reviews : Resident Evil 4 (2023) (0)


In [9]:
# ============================================================
# Cell 9 — Save data ra CSV
# ============================================================

reviews_path = f'{DATA_DIR}/raw_steam_data.csv'
games_path   = f'{DATA_DIR}/raw_games_info.csv'

reviews_df.to_csv(reviews_path, index=False, encoding='utf-8-sig')
games_df.to_csv(games_path,   index=False, encoding='utf-8-sig')

print('=== Saved files ===')
for path in [reviews_path, games_path]:
    abs_path = os.path.abspath(path)
    size_kb  = os.path.getsize(abs_path) / 1024
    print(f'  {path}')
    print(f'  → {abs_path}')
    print(f'  → {size_kb:.1f} KB')
    print()

print('Task 1 complete! Moving to 02_eda_analysis.ipynb')

=== Saved files ===
  ../data/raw_steam_data.csv
  → D:\qkhenh\Project\CodeAlpha_DataAnalytics\data\raw_steam_data.csv
  → 1612.3 KB

  ../data/raw_games_info.csv
  → D:\qkhenh\Project\CodeAlpha_DataAnalytics\data\raw_games_info.csv
  → 2.4 KB

Task 1 complete! Moving to 02_eda_analysis.ipynb
